In [5]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F

# 1. Load the Delta tables from Lakehouse

staged_fact_df = spark.read.table("stg_daily_account_balance")
dim_customer_df = spark.read.table("dim_customer")
dim_account_df = spark.read.table("dim_account")
dim_branch_df = spark.read.table("dim_branch")
dim_product_df = spark.read.table("dim_product")

# 2. Perform Left Joins and handle missing keys using F.coalesce
# F.coalesce takes the first non-null value it finds. If the dim key is null, it falls back to -1.
final_fact_df = staged_fact_df \
    .join(dim_customer_df, staged_fact_df["customer_key"] == dim_customer_df["customer_key"], "left") \
    .join(dim_account_df, staged_fact_df["account_key"] == dim_account_df["account_key"], "left") \
    .join(dim_branch_df, staged_fact_df["branch_key"] == dim_branch_df["branch_key"], "left") \
    .join(dim_product_df, staged_fact_df["product_key"] == dim_product_df["product_key"], "left") \
    .select(
        staged_fact_df["date_key"],
        F.coalesce(dim_branch_df["branch_key"], F.lit(-1)).alias("branch_key"),
        F.coalesce(dim_product_df["product_key"], F.lit(-1)).alias("product_key"),
        # If the dimension join fails, the dim key is null -> replaced with -1
        F.coalesce(dim_customer_df["customer_key"], F.lit(-1)).alias("customer_key"),
        F.coalesce(dim_account_df["account_key"], F.lit(-1)).alias("account_key"),
        staged_fact_df["opening_balance"],
        staged_fact_df["debit_amount"],
        staged_fact_df["credit_amount"],
        staged_fact_df["closing_balance"],
        staged_fact_df["transaction_count"]
    )

# 3. Save the treated data into the final Delta table
# Use 'append' or 'overwrite' depending on your pipeline logic
final_fact_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_ccount_balance")

StatementMeta(, 45d45529-7fd3-45de-a41c-df31750e6599, 7, Finished, Available, Finished, False)

In [7]:
df = spark.sql("SELECT count(*) FROM lh_fabric.dbo.gold_fact_ccount_balance ")
display(df)

StatementMeta(, 45d45529-7fd3-45de-a41c-df31750e6599, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a121992d-0d0a-4a24-8c3f-78d9744bb081)